### Running the Experiment

This experiment is implemented as a single end-to-end execution block to facilitate reproducible execution with minimal manual setup. The script covers the complete pipeline: dependency installation, seed initialization, dataset loading, preprocessing, training, evaluation, checkpointing, hyperparameter search, and final model export.


### Pipeline Overview

The script performs the following steps:

- installs required libraries (`transformers`, `datasets`, `accelerate`, `scikit-learn`);
- automatically selects the computation device (GPU if available, otherwise CPU);
- sets random seeds for reproducibility;
- loads the pretrained model `microsoft/codebert-base`;
- downloads the SemEval-2026 Task 13 dataset;
- filters data by programming language for cross-language experiments;
- tokenizes inputs and builds PyTorch dataloaders;
- trains multiple configurations from a hyperparameter grid;
- evaluates using macro F1-score;
- saves checkpoints and resumes training if available;
- stores results in a JSON file and exports the best checkpoint corresponding to the best-performing run.


### Experimental Setup

The experiment uses **CodeBERT** for binary classification on code snippets.  
Training follows a cross-language rotation scheme:

- **Java + Python → validation on C++**
- **C++ + Python → validation on Java**
- **C++ + Java → validation on Python**

This setup evaluates generalization across programming languages.


### Hyperparameter Configurations

Five configurations are evaluated:

**Config 0**
- batch size: 16  
- epochs: 6  
- lr (encoder): 1e-5  
- lr (classifier): 1e-4  
- warmup: 0.08  
- initial unfreeze: 2  
- step: 2  
- early stop: 2  

**Config 1**
- batch size: 16  
- epochs: 8  
- lr (encoder): 5e-6  
- lr (classifier): 5e-5  
- warmup: 0.10  
- initial unfreeze: 2  
- step: 2  
- early stop: 2  

**Config 2**
- batch size: 32  
- epochs: 5  
- lr (encoder): 2e-5  
- lr (classifier): 2e-4  
- warmup: 0.06  
- initial unfreeze: 4  
- step: 4  
- early stop: 2  

**Config 3**
- batch size: 32  
- epochs: 6  
- lr (encoder): 1e-5  
- lr (classifier): 1e-4  
- warmup: 0.06  
- initial unfreeze: 4  
- step: 2  
- early stop: 2  

**Config 4**
- batch size: 16  
- epochs: 6  
- lr (encoder): 1e-5  
- lr (classifier): 5e-5  
- warmup: 0.12  
- initial unfreeze: 2  
- step: 2  
- early stop: 2  

Each configuration is evaluated across all rotations, and the final score is the **mean macro F1**.


### Training Details

- **partial gradual unfreezing of the encoder** (partial encoder unfreezing is applied, with fixed optimizer parameter groups);
- AdamW optimizer with separate learning rates for encoder and classifier;
- linear scheduler with warmup;
- mixed precision enabled on GPU;
- gradient clipping for stability (applied during mixed precision training);
- evaluation after each epoch using macro F1.


### Checkpointing

After every epoch, the script saves:
- model state  
- optimizer  
- scheduler  
- scaler (if GPU)  
- training metadata  

If a checkpoint exists, training automatically resumes from the last saved checkpoint.


### Runtime Considerations

- GPU is recommended; CPU execution may be slow;
- dataset and model are downloaded automatically at first run;
- batch size can be reduced if memory issues occur;
- validation may take noticeable time;
- multiple checkpoint files will be created;
- exact numerical results may vary slightly across runs due to stochastic training dynamics and hardware differences.

In [ ]:
import os
import json
import torch
import random
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

!pip install -q transformers datasets accelerate scikit-learn

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
samanta = 42

def seteaza_samanta(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if dispozitiv.type == "cuda":
        torch.cuda.manual_seed_all(s)

seteaza_samanta(samanta)

nume_model = "microsoft/codebert-base"
tokenizator = AutoTokenizer.from_pretrained(nume_model)
lungime_max = 256

dataset_initial = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
date_antrenare_totale = dataset_initial["train"]
date_validare_totale = dataset_initial["validation"]

print("Limbaje disponibile:", sorted(set(date_antrenare_totale["language"])))

rotatii_limbaje = [
    (["Java", "Python"], "C++"),
    (["C++", "Python"], "Java"),
    (["C++", "Java"], "Python"),
]
def tokenizeaza(exemple):
    t = tokenizator(
        exemple["code"],
        truncation=True,
        padding="max_length",
        max_length=lungime_max,
    )
    t["labels"] = exemple["label"]
    return t

def filtreaza_dupa_limbaj(dataset, limbaje):
    if isinstance(limbaje, str):
        limbaje = [limbaje]
    return dataset.filter(lambda x: x["language"] in limbaje)

def construieste_loader(dataset, marime_batch, shuffle):
    proc = dataset.map(
        tokenizeaza,
        batched=True,
        remove_columns=[c for c in dataset.column_names if c not in ["code","label","language","generator"]],
    )
    proc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
    return DataLoader(proc, batch_size=marime_batch, shuffle=shuffle)

def ingheata_encoder(model):
    for p in model.roberta.parameters():
        p.requires_grad = False

def deblocheaza_ultimele_straturi(model, nr):
    straturi = list(model.roberta.encoder.layer)
    total = len(straturi)
    start = max(0, total - nr)
    for i, strat in enumerate(straturi):
        permite = i >= start
        for p in strat.parameters():
            p.requires_grad = permite

@torch.no_grad()
def evalueaza_f1(model, loader_valid):
    model.eval()
    toate_pred, toate_adev = [], []
    for lot in loader_valid:
        intrari = lot["input_ids"].to(dispozitiv)
        masca = lot["attention_mask"].to(dispozitiv)
        etichete = lot["labels"].to(dispozitiv)

        iesiri = model(input_ids=intrari, attention_mask=masca).logits
        pred = torch.argmax(iesiri, dim=1)

        toate_pred += pred.cpu().tolist()
        toate_adev += etichete.cpu().tolist()

    return f1_score(toate_adev, toate_pred, average="macro")
def salveaza_checkpoint(cale, payload):
    os.makedirs(os.path.dirname(cale), exist_ok=True)
    torch.save(payload, cale)

def incarca_checkpoint(cale):
    if os.path.exists(cale):
        return torch.load(cale, map_location=dispozitiv)
    return None
def incarca_rezultate(cale_json):
    if os.path.exists(cale_json):
        with open(cale_json, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"runs": {}, "cfg_means": {}, "best": {"cfg_idx": None, "config": None, "mean_f1": -1, "best_run_key": None}}

def salveaza_rezultate(cale_json, rezultate):
    os.makedirs(os.path.dirname(cale_json), exist_ok=True)
    tmp = cale_json + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(rezultate, f, ensure_ascii=False, indent=2)
    os.replace(tmp, cale_json)

def cheie_run(idx_cfg, idx_rot, limbaj_val):
    return f"cfg{idx_cfg}_rot{idx_rot}_val{limbaj_val}"
def ruleaza_o_antrenare_cu_checkpoint(date_train, date_val, hiper, cale_checkpoint, meta_tag=""):
    model = AutoModelForSequenceClassification.from_pretrained(
        nume_model, num_labels=2
    ).to(dispozitiv)

    loader_train = construieste_loader(date_train, hiper["marime_batch"], True)
    loader_valid = construieste_loader(date_val, hiper["marime_batch"], False)

    ingheata_encoder(model)
    deblocheaza_ultimele_straturi(model, hiper["deblocare_initiala"])

    param_enc, param_clf = [], []
    for nume, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in nume:
            param_clf.append(p)
        else:
            param_enc.append(p)

    optimizator = AdamW(
        [
            {"params": param_enc, "lr": hiper["lr_encoder"]},
            {"params": param_clf, "lr": hiper["lr_clasificator"]},
        ]
    )

    pasi_total = len(loader_train) * hiper["epoci"]
    pasi_warmup = int(pasi_total * hiper["warmup_ratio"])

    scheduler = get_linear_schedule_with_warmup(
        optimizator, pasi_warmup, pasi_total
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(dispozitiv.type == "cuda"))

    cel_mai_bun_f1 = 0.0
    fara_impro = 0
    start_epoca = 0

    ckpt = incarca_checkpoint(cale_checkpoint)
    if ckpt is not None:
        model.load_state_dict(ckpt["model_state"])
        optimizator.load_state_dict(ckpt["opt_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if dispozitiv.type == "cuda" and ckpt.get("scaler_state") is not None:
            scaler.load_state_dict(ckpt["scaler_state"])

        cel_mai_bun_f1 = ckpt.get("cel_mai_bun_f1", 0.0)
        fara_impro = ckpt.get("fara_impro", 0)
        start_epoca = ckpt.get("epoca", -1) + 1

        print(f"[{meta_tag}] Am gasit checkpoint. Reiau de la epoca {start_epoca + 1}. Best F1={cel_mai_bun_f1:.4f}")
    else:
        print(f"[{meta_tag}] Nu am gasit checkpoint. Pornesc de la epoca 1.")

    for epoca in range(start_epoca, hiper["epoci"]):
        deblocheaza_ultimele_straturi(
            model,
            hiper["deblocare_initiala"] + epoca * hiper["pas_deblocare"]
        )

        model.train()
        pierderi = []

        for lot in loader_train:
            optimizator.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(dispozitiv.type == "cuda")):
                ies = model(
                    input_ids=lot["input_ids"].to(dispozitiv),
                    attention_mask=lot["attention_mask"].to(dispozitiv),
                    labels=lot["labels"].to(dispozitiv)
                )
                pierdere = ies.loss

            scaler.scale(pierdere).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizator)
            scaler.update()
            scheduler.step()

            pierderi.append(float(pierdere.item()))

        f1 = evalueaza_f1(model, loader_valid)
        pierdere_medie = float(np.mean(pierderi)) if pierderi else 0.0
        print(f"[{meta_tag}] Epoca {epoca+1}/{hiper['epoci']} | loss={pierdere_medie:.4f} | F1={f1:.4f} | best={cel_mai_bun_f1:.4f}")

        if f1 > cel_mai_bun_f1:
            cel_mai_bun_f1 = f1
            fara_impro = 0
        else:
            fara_impro += 1

        payload = {
            "epoca": epoca,
            "model_state": model.state_dict(),
            "opt_state": optimizator.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict() if dispozitiv.type == "cuda" else None,
            "cel_mai_bun_f1": cel_mai_bun_f1,
            "fara_impro": fara_impro,
            "meta": {"hiper": hiper, "meta_tag": meta_tag}
        }
        salveaza_checkpoint(cale_checkpoint, payload)

        if fara_impro >= hiper["early_stop"]:
            print(f"[{meta_tag}] Early stop: fara imbunatatire {fara_impro} epoci la rand.")
            break

    return cel_mai_bun_f1

grid_hiperparametri = [
    {
        "marime_batch": 16, "epoci": 6, "lr_encoder": 1e-5, "lr_clasificator": 1e-4,
        "warmup_ratio": 0.08, "deblocare_initiala": 2, "pas_deblocare": 2, "early_stop": 2
    },
    {
        "marime_batch": 16, "epoci": 8, "lr_encoder": 5e-6, "lr_clasificator": 5e-5,
        "warmup_ratio": 0.10, "deblocare_initiala": 2, "pas_deblocare": 2, "early_stop": 2
    },
    {
        "marime_batch": 32, "epoci": 5, "lr_encoder": 2e-5, "lr_clasificator": 2e-4,
        "warmup_ratio": 0.06, "deblocare_initiala": 4, "pas_deblocare": 4, "early_stop": 2
    },
    {
        "marime_batch": 32, "epoci": 6, "lr_encoder": 1e-5, "lr_clasificator": 1e-4,
        "warmup_ratio": 0.06, "deblocare_initiala": 4, "pas_deblocare": 2, "early_stop": 2
    },
    {
        "marime_batch": 16, "epoci": 6, "lr_encoder": 1e-5, "lr_clasificator": 5e-5,
        "warmup_ratio": 0.12, "deblocare_initiala": 2, "pas_deblocare": 2, "early_stop": 2
    },
]

baza_ckpt = "./ckpt_grid_fresh"
cale_rezultate = os.path.join(baza_ckpt, "results.json")
rezultate = incarca_rezultate(cale_rezultate)

print("Rezultate incarcate:", len(rezultate["runs"]), "rulari salvate deja.")
if rezultate["best"]["cfg_idx"] is not None:
    print("BEST salvat (din results.json):", rezultate["best"])

cel_mai_bun_scor = -1
cea_mai_buna_config = None

for idx_cfg, config in enumerate(grid_hiperparametri):
    scoruri = []
    print("\n==========================")
    print("Config:", config)

    for idx_rot, (limbaje_train, limbaj_val) in enumerate(rotatii_limbaje):
        date_train = filtreaza_dupa_limbaj(date_antrenare_totale, limbaje_train)
        date_val = filtreaza_dupa_limbaj(date_validare_totale, limbaj_val)

        meta_tag = f"cfg{idx_cfg}_rot{idx_rot}_val{limbaj_val}"
        cale_checkpoint = os.path.join(baza_ckpt, f"{meta_tag}.pt")

        f1 = ruleaza_o_antrenare_cu_checkpoint(
            date_train, date_val, config,
            cale_checkpoint=cale_checkpoint,
            meta_tag=meta_tag
        )

        scoruri.append(f1)
        print(f"Train={limbaje_train} | Val={limbaj_val} | F1(best)={f1:.4f}")
        run_key = cheie_run(idx_cfg, idx_rot, limbaj_val)
        rezultate["runs"][run_key] = {
            "cfg_idx": idx_cfg,
            "rot_idx": idx_rot,
            "val_language": limbaj_val,
            "train_languages": limbaje_train,
            "config": config,
            "best_f1": float(f1),
            "checkpoint_path": cale_checkpoint
        }
        salveaza_rezultate(cale_rezultate, rezultate)

    medie = float(np.mean(scoruri))
    print("F1 mediu:", medie)

    rezultate["cfg_means"][f"cfg{idx_cfg}"] = {
        "config": config,
        "mean_f1": float(medie),
        "per_rotation_best_f1": [float(x) for x in scoruri],
    }
    if medie > rezultate["best"]["mean_f1"]:
        best_run_key = None
        best_run_f1 = -1
        for k, v in rezultate["runs"].items():
            if v["cfg_idx"] == idx_cfg and v["best_f1"] > best_run_f1:
                best_run_f1 = v["best_f1"]
                best_run_key = k

        rezultate["best"] = {
            "cfg_idx": idx_cfg,
            "config": config,
            "mean_f1": float(medie),
            "best_run_key": best_run_key
        }

    salveaza_rezultate(cale_rezultate, rezultate)
    if medie > cel_mai_bun_scor:
        cel_mai_bun_scor = medie
        cea_mai_buna_config = config

print("\n==========================")
print("CEA MAI BUNA CONFIG (din runda curenta):", cea_mai_buna_config)
print("CEL MAI BUN F1 MEDIU (din runda curenta):", cel_mai_bun_scor)

print("\n==========================")
print("CEA MAI BUNA CONFIG (din results.json):", rezultate["best"]["config"])
print("CEL MAI BUN F1 MEDIU (din results.json):", rezultate["best"]["mean_f1"])
print("BEST cfg_idx (din results.json):", rezultate["best"]["cfg_idx"])
print("Fisier rezultate:", cale_rezultate)
best_run_key = rezultate["best"].get("best_run_key", None)
if best_run_key is None:
    print("\nNu exista best_run_key in results.json (probabil nu s-a terminat nicio configuratie complet).")
else:
    best_run = rezultate["runs"][best_run_key]
    cale_ckpt_best = best_run["checkpoint_path"]
    print("\n==========================")
    print("Incarc BEST model din checkpoint:", cale_ckpt_best)
    print("Best run:", best_run_key, "| best_f1:", best_run["best_f1"], "| val:", best_run["val_language"])

    ckpt_best = torch.load(cale_ckpt_best, map_location=dispozitiv)

    best_model = AutoModelForSequenceClassification.from_pretrained(
        nume_model, num_labels=2
    )
    best_model.load_state_dict(ckpt_best["model_state"])
    best_model.to(dispozitiv)
    best_model.eval()
    cale_best_model = os.path.join(baza_ckpt, "BEST_MODEL.pt")
    torch.save(
        {
            "model_state": best_model.state_dict(),
            "best": rezultate["best"],
            "best_run": best_run,
            "base_model": nume_model,
            "num_labels": 2
        },
        cale_best_model
    )
    print("Salvat BEST_MODEL.pt la:", cale_best_model)


### Expected Output (Key Reproducibility Markers)

During execution, the notebook prints detailed logs for dataset loading, training progress, checkpoint loading/resume status, and final best-model selection.  
Some non-critical warnings may also appear (for example about a missing `HF_TOKEN` in Colab or deprecated AMP API calls in PyTorch). These do not affect the correctness of the experiment.  
Because training involves randomness, hardware differences, and early stopping, the exact loss and Macro F1 values may vary slightly across runs.  
However, the following key outputs should be observed:

- The dataset is downloaded and loaded successfully.
- The available languages are displayed as:
  `['C++', 'Java', 'Python']`
- Multiple configurations are evaluated in a small grid-search procedure.
- For each rotation, the script prints:
  - the validation language,
  - the epoch-wise training loss,
  - the validation Macro F1,
  - the best score achieved so far,
  - and whether early stopping was triggered.
- At the end, the script reports:
  - the best configuration from the current run,
  - the best configuration stored in `results.json`,
  - the best configuration index,
  - the selected best checkpoint,
  - and the saved final model path.

A representative summary of the final output is shown below:

```text
Limbaje disponibile: ['C++', 'Java', 'Python']

CEA MAI BUNA CONFIG (din runda curenta):
{'marime_batch': 32, 'epoci': 6, 'lr_encoder': 1e-05,
 'lr_clasificator': 0.0001, 'warmup_ratio': 0.06,
 'deblocare_initiala': 4, 'pas_deblocare': 2, 'early_stop': 2}

CEL MAI BUN F1 MEDIU (din runda curenta): 0.6928

CEA MAI BUNA CONFIG (din results.json):
{'marime_batch': 32, 'epoci': 6, 'lr_encoder': 1e-05,
 'lr_clasificator': 0.0001, 'warmup_ratio': 0.06,
 'deblocare_initiala': 4, 'pas_deblocare': 2, 'early_stop': 2}

CEL MAI BUN F1 MEDIU (din results.json): 0.6928
BEST cfg_idx (din results.json): 3

Incarc BEST model din checkpoint: ./ckpt_grid_fresh/cfg3_rot1_valJava.pt
Best run: cfg3_rot1_valJava | best_f1: 0.9033 | val: Java
Salvat BEST_MODEL.pt la: ./ckpt_grid_fresh/BEST_MODEL.pt


Note that the runtime logs and final printed outputs are displayed in Romanian, as the notebook messages were intentionally written in Romanian for consistency with the rest of the project.

In [ ]:
import json

path = "./ckpt_grid_fresh/results.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

data


{'runs': {'cfg0_rot0_valC++': {'cfg_idx': 0,
   'rot_idx': 0,
   'val_language': 'C++',
   'train_languages': ['Java', 'Python'],
   'config': {'marime_batch': 16,
    'epoci': 6,
    'lr_encoder': 1e-05,
    'lr_clasificator': 0.0001,
    'warmup_ratio': 0.08,
    'deblocare_initiala': 2,
    'pas_deblocare': 2,
    'early_stop': 2},
   'best_f1': 0.8219794052689754,
   'checkpoint_path': './ckpt_grid_fresh/cfg0_rot0_valC++.pt'},
  'cfg0_rot1_valJava': {'cfg_idx': 0,
   'rot_idx': 1,
   'val_language': 'Java',
   'train_languages': ['C++', 'Python'],
   'config': {'marime_batch': 16,
    'epoci': 6,
    'lr_encoder': 1e-05,
    'lr_clasificator': 0.0001,
    'warmup_ratio': 0.08,
    'deblocare_initiala': 2,
    'pas_deblocare': 2,
    'early_stop': 2},
   'best_f1': 0.9033660012869051,
   'checkpoint_path': './ckpt_grid_fresh/cfg0_rot1_valJava.pt'},
  'cfg0_rot2_valPython': {'cfg_idx': 0,
   'rot_idx': 2,
   'val_language': 'Python',
   'train_languages': ['C++', 'Java'],
   'config'